# Week 5: Deep Learning and Global Models
## In-Class Exercises

**Objective.** Fit a neural forecaster, understand what "global" buys you, and find the point where the extra machinery stops paying for itself.

> **Before you start:** switch the Colab runtime to a GPU (Runtime -> Change runtime type -> T4). CPU works too, just slower. The install cell takes 2-3 minutes, so start it now.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | NHITS on a single series, and why the result is underwhelming. |
| 2 | Blanks we fill in together | A panel of 60 series: one global model vs. 60 local ones. |
| 3 | On your own, ~15 min | Change the architecture and the input window, and measure what moved. |

In [ ]:
!pip install -q neuralforecast statsforecast

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, NBEATS
from neuralforecast.utils import AirPassengersDF
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)


def mae(y, yhat):
    return np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float)))

def mase(y, yhat, y_train, m=12):
    y_train = np.asarray(y_train, float)
    return mae(y, yhat) / np.mean(np.abs(y_train[m:] - y_train[:-m]))

AirPassengersDF.head()

---
## Part 1. NHITS on one short series

The Nixtla ecosystem uses one long format everywhere: `unique_id`, `ds`, `y`, one row per series per timestamp. Learn it once and every library in the stack accepts your data.

NHITS is a stack of MLP blocks at different sampling rates, so each block specializes in a frequency band. Fast, strong on benchmarks, and about to struggle on the 144-point series below. Watch the comparison against `AutoETS`.

In [ ]:
h, m = 12, 12
Y = AirPassengersDF.copy()
Y_train = Y.iloc[:-h]
Y_test = Y.iloc[-h:]

nf = NeuralForecast(
    models=[NHITS(h=h, input_size=2 * h, max_steps=200, random_seed=1)],
    freq="MS",
)
nf.fit(df=Y_train)
nn_fc = nf.predict()
nn_fc.head()

In [ ]:
sf = StatsForecast(models=[AutoETS(season_length=m), SeasonalNaive(season_length=m)], freq="MS")
stat_fc = sf.forecast(df=Y_train, h=h)

merged = (Y_test.merge(nn_fc, on=["unique_id", "ds"])
                .merge(stat_fc, on=["unique_id", "ds"]))

ax = Y_train.set_index("ds")["y"].tail(48).plot(color="black", label="train")
merged.set_index("ds")[["y", "NHITS", "AutoETS", "SeasonalNaive"]].plot(ax=ax)
ax.legend(fontsize=8)
ax.set_title("One series, 144 observations")
plt.show()

for col in ["NHITS", "AutoETS", "SeasonalNaive"]:
    print(f"{col:14s} MASE: {mase(merged['y'], merged[col], Y_train['y'], m):.3f}")

**Notice:**

The network has thousands of parameters and 132 training observations. `AutoETS` has about a dozen parameters and a structural assumption that is true for this series. The small model with the right structure wins, which is neither a surprise nor a scandal.

Deep learning models are not "better forecasters." They **trade structural assumptions for data**. Give one 132 observations and it has nothing to trade. Give it 60 series of 132 observations and the trade pays, because a shared model learns the seasonal shape from all of them at once.

That is the argument for **global** models. Part 2 tests it.

---
## Part 2. Sixty series, one model

The panel below has three clusters of behavior plus short series with barely enough history to fit anything locally. Those short series are where global models earn their reputation.

In [ ]:
def make_panel(n_per_group=20, n=72, seed=5):
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2015-01-01")
    for g, (trend, amp, noise) in enumerate([(0.4, 12, 3.0), (0.0, 25, 5.0), (-0.2, 5, 2.0)]):
        for i in range(n_per_group):
            uid = f"g{g}_s{i:02d}"
            # short series: the last third of each group has only 30 observations
            length = n if i < (2 * n_per_group) // 3 else 30
            t = np.arange(length)
            phase = rng.uniform(0, 2 * np.pi)
            level = rng.uniform(80, 160)
            y = (level + trend * t
                 + amp * np.sin(2 * np.pi * t / 12 + phase)
                 + rng.normal(scale=noise, size=length))
            rows.append(pd.DataFrame({
                "unique_id": uid,
                "ds": pd.date_range(start, periods=length, freq="MS"),
                "y": y,
            }))
    return pd.concat(rows, ignore_index=True)


panel = make_panel()
print(panel.groupby("unique_id").size().describe()[["count", "min", "max"]])

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, uid in zip(axes, ["g0_s00", "g1_s00", "g2_s19"]):
    d = panel.query("unique_id == @uid")
    ax.plot(d["ds"], d["y"])
    ax.set_title(uid)
plt.tight_layout()
plt.show()

In [ ]:
H = 12
panel_train = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[:-H])
panel_test = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[-H:])

# Local models: one fit per series.
sf = StatsForecast(models=[AutoETS(season_length=12), SeasonalNaive(season_length=12)], freq="MS", n_jobs=-1)
local_fc = sf.forecast(df=panel_train, h=H)
local_fc.head()

In [ ]:
# TODO - fit ONE global NHITS across every series in the panel and predict H steps.
#   Hint: the call is identical to Part 1. NeuralForecast infers the panel from `unique_id`.
#   Use NHITS(h=H, input_size=2 * H, max_steps=300, random_seed=1).
nf_global = ...
global_fc = ...

<details>
<summary><b>Show the lines</b></summary>

```python
nf_global = NeuralForecast(models=[NHITS(h=H, input_size=2 * H, max_steps=300, random_seed=1)], freq="MS")
nf_global.fit(df=panel_train)
global_fc = nf_global.predict()
```

Note what you did *not* write: no loop over series, no per-series hyperparameters, no special handling for the 30-observation series.
</details>

In [ ]:
ev = (panel_test.merge(local_fc, on=["unique_id", "ds"])
                .merge(global_fc, on=["unique_id", "ds"]))

scales = (panel_train.groupby("unique_id")["y"]
          .apply(lambda s: np.mean(np.abs(s.to_numpy()[12:] - s.to_numpy()[:-12]))))

def per_series_mase(df, col):
    err = df.assign(e=lambda d: (d["y"] - d[col]).abs()).groupby("unique_id")["e"].mean()
    return err / scales

per_series = pd.DataFrame({c: per_series_mase(ev, c) for c in ["NHITS", "AutoETS", "SeasonalNaive"]})
lengths = panel_train.groupby("unique_id").size().rename("n_obs")

summary = pd.DataFrame({
    "mean MASE": per_series.mean(),
    "median MASE": per_series.median(),
    "worst 10% mean": per_series.apply(lambda c: c.nlargest(6).mean()),
    "win rate": (per_series.idxmin(axis=1).value_counts(normalize=True)),
})
summary.round(3)

In [ ]:
# Where does global help? Split by series length.
joined = per_series.join(lengths)
joined.groupby(joined["n_obs"] < 72).mean().rename(index={False: "long series", True: "short series"}).round(3)

**Reading the two tables.**

1. On long series, `AutoETS` is competitive or better. The structure it assumes is the structure we generated.
2. On **short** series the global model pulls ahead, sometimes sharply. `AutoETS` has 18 usable observations and cannot identify a seasonal pattern; NHITS learned the shape from the other 59 series.
3. In practice the `worst 10%` column matters more than the mean. A method with a slightly worse average and a much better tail is often the one you ship, because the tail generates the angry emails.

---
## Part 3. Make it better, and say what moved

About 15 minutes. Turn one knob at a time; turning three at once teaches you nothing.

**Tasks.**

1. Re-fit with `input_size` of `H`, `2*H`, and `4*H`. Report the winner and whether it differs between long and short series.
2. Add `NBEATS(h=H, input_size=2*H, max_steps=300)` alongside NHITS. Compare, then average the two and compare again.
3. Raise `max_steps` from 300 to 1000 on the best configuration and plot MASE against training steps. Note where it stops improving and whether it degrades.
4. In two sentences: which knob mattered most, and what you would try with another 30 minutes of compute.

Keep a results table. "I tried some things and the second one seemed better" is not a finding.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Hints and what to expect</b></summary>

```python
results = []
for size_mult in [1, 2, 4]:
    nf_i = NeuralForecast(
        models=[NHITS(h=H, input_size=size_mult * H, max_steps=300, random_seed=1)], freq="MS")
    nf_i.fit(df=panel_train)
    f = nf_i.predict()
    e = panel_test.merge(f, on=["unique_id", "ds"])
    results.append({"input_size": size_mult * H,
                    "mean MASE": per_series_mase(e, "NHITS").mean()})
pd.DataFrame(results)
```

Expect `2*H` to be a reasonable default, and `4*H` to help long series while hurting short ones: a longer input window disqualifies short series from contributing training windows at all. That interaction between window length and series length never appears in tutorials and always appears in production.
</details>

---
## Wrap-up

1. **Global models trade structure for data.** With one short series there is nothing to trade.
2. **The gain concentrates in short series and the error tail**, not the mean.
3. **`input_size` interacts with series length.** Longer windows silently discard short series.
4. **Zero-shot works when the new series resembles the training panel.** Verify that before believing the number.

Next week: probabilistic forecasts, and making a hierarchy add up.